# 🍅 Crop Disease Detection using CNN
### Tomato leaf disease classification with a Convolutional Neural Network (PlantVillage)

**Task type:** Computer Vision — multi-class image classification (10 classes)
**Framework:** TensorFlow / Keras
**Notebook flow:** Problem → Data audit → EDA → Pipeline → Custom CNN → Transfer learning → Evaluation → Explainability → Deployment helper → Conclusion

---
## Table of contents
1. Project title
2. Problem statement & objectives
3. Environment setup & configuration
4. Dataset loading & audit (integrity, duplicates, leakage)
5. Exploratory data analysis
6. Train / validation / test split
7. Input pipeline & data augmentation
8. Model 1 — Custom CNN
9. Model 2 — Transfer learning (MobileNetV2)
10. Model selection & test evaluation
11. Error analysis
12. Explainability (Grad-CAM)
13. Save artefacts & inference helper
14. Conclusion, limitations & future work

## SECTION 2 — Problem statement & objectives

Tomato plants are affected by a range of fungal, bacterial, viral and pest-related diseases. Identifying the disease early from a photograph of a leaf lets growers act sooner and use treatment more selectively.

**Goal:** given an RGB photograph of a single tomato leaf, predict which of 10 categories it belongs to (9 diseases/pests + healthy).

**Objectives**
1. Audit the dataset (class balance, image size, corrupt files, duplicates, train/test leakage).
2. Build and train a CNN from scratch as a baseline.
3. Compare against a transfer-learning model (MobileNetV2, ImageNet weights).
4. Evaluate honestly on data the model never saw during training or model selection (accuracy, macro precision/recall/F1, confusion matrix, per-class errors).
5. Explain predictions with Grad-CAM to check the model looks at the lesion, not the background.
6. Provide a reusable `predict_image()` function.

**Target column:** the class label is the *name of the sub-folder* (`Tomato___Bacterial_spot`, …). There is no CSV — the dataset is an image-folder dataset.

> **Note on the supplied `cnn_train.py`:** it targets the legacy Keras-1 API (`Convolution2D(32, 3, 3)`, `output_dim`, `fit_generator`) which no longer runs on current TensorFlow, trains for `epochs=10000`, and uses the validation folder for monitoring only. This notebook keeps its core idea (128×128 input, Conv→Pool blocks, Dense-128 head, Dropout 0.5, Adam) but modernises and extends it.

## SECTION 3 — Environment setup & configuration

In [ ]:
# Uncomment if running on a fresh environment (e.g. Colab already has all of these):
# !pip install -q tensorflow scikit-learn pandas matplotlib seaborn pillow

In [ ]:
import os, re, json, time, random, hashlib, zipfile, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

print("TensorFlow :", tf.__version__)
print("GPUs found :", tf.config.list_physical_devices("GPU"))

In [ ]:
# ──────── Global configuration ────────
SEED        = 42
IMG_SIZE    = (128, 128)      # 128x128 input size
BATCH_SIZE  = 32
VAL_FRACTION_FOLDS = 10       # 1 of 10 grouped folds -> ~10% validation

QUICK_RUN       = False
EPOCHS_CNN      = 2 if QUICK_RUN else 30
EPOCHS_TL_HEAD  = 2 if QUICK_RUN else 8
EPOCHS_TL_FT    = 2 if QUICK_RUN else 10

# Dynamic local dataset discovery
def find_dataset_folders():
    candidates = [
        (Path('tomato/train'), Path('tomato/val')),
        (Path('tomato/train'), Path('tomato/valid')),
        (Path('train'), Path('valid')),
        (Path('train'), Path('val')),
        (Path('../train'), Path('../valid')),
        (Path('../train'), Path('../val')),
        (Path('dataset/train'), Path('dataset/val')),
        (Path('dataset/train'), Path('dataset/valid')),
    ]
    for tr, va in candidates:
        if tr.is_dir() and va.is_dir():
            return tr, va
    raise FileNotFoundError('Could not automatically locate dataset folders!')

TRAIN_DIR, VAL_DIR = find_dataset_folders()
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print('Detected Train folder   :', TRAIN_DIR.resolve())
print('Detected Val/Test folder:', VAL_DIR.resolve())


## SECTION 4 — Dataset loading & audit

The dataset ships with two folders, `train/` and `val/`, each containing one sub-folder per class.
Before modelling we build a single table (`path`, `class`, `split`) and audit it for:

* **File types & corrupt images**
* **Image size consistency**
* **Exact duplicate files** (same bytes) — inside train and *across* train/val
* **Near-duplicate "source leaf" groups.** PlantVillage contains augmented copies of the same photograph (file names contain `…_original_<uuid>___…` and the source `<uuid>`). If copies of one leaf land in both training and validation data, accuracy is inflated. We extract that source id so we can split *by source leaf* later.

In [ ]:
VALID_EXT = {'.jpg', '.jpeg', '.png'}
UUID_RE = re.compile(r'([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})___')

def source_id(path):
    '''UUID of the original photo (first uuid followed by '___'); falls back to file path.'''
    m = UUID_RE.search(Path(path).name)
    return m.group(1) if m else str(path)

def scan_split(folder_path, split_name):
    rows = []
    for cls_dir in sorted(p for p in folder_path.iterdir() if p.is_dir()):
        for f in sorted(cls_dir.iterdir()):
            if f.suffix.lower() in VALID_EXT:
                rows.append({'path': str(f), 'class': cls_dir.name, 'split': split_name})
    return pd.DataFrame(rows)

df_train_full = scan_split(TRAIN_DIR, 'train')
df_test       = scan_split(VAL_DIR,   'test')      # held-out as FINAL TEST set

class_names = sorted(df_train_full['class'].unique())
assert class_names == sorted(df_test['class'].unique()), 'train/val class folders differ!'
NUM_CLASSES = len(class_names)
class_to_idx = {c: i for i, c in enumerate(class_names)}
pretty = lambda c: c.replace('Tomato___', '').replace('_', ' ').replace('  ', ' ')
pretty_names = [pretty(c) for c in class_names]

for d in (df_train_full, df_test):
    d['label'] = d['class'].map(class_to_idx)
    d['group'] = d['path'].map(source_id)

print(f'Task            : multi-class image classification, {NUM_CLASSES} classes')
print(f'Train images    : {len(df_train_full):,}')
print(f'Test images     : {len(df_test):,}   (from provided validation/test folder)')
print(f'Total images    : {len(df_train_full)+len(df_test):,}')
print('\nClasses:')
for i, c in enumerate(class_names): print(f'  {i}: {c}')


In [ ]:
# ───── Integrity + size check on EVERY file ─────
def inspect_image(p):
    try:
        with Image.open(p) as im:
            size, mode = im.size, im.mode
            im.verify()
        return size, mode, True
    except Exception:
        return None, None, False

all_df = pd.concat([df_train_full, df_test], ignore_index=True)
t0 = time.time()
info = [inspect_image(p) for p in all_df["path"]]
all_df["size"]  = [i[0] for i in info]
all_df["mode"]  = [i[1] for i in info]
all_df["valid"] = [i[2] for i in info]
print(f"Inspected {len(all_df):,} files in {time.time()-t0:.1f}s")

print("\nCorrupt / unreadable files :", int((~all_df["valid"]).sum()))
print("Image sizes (w, h)         :", dict(Counter(all_df.loc[all_df.valid, "size"])))
print("Colour modes               :", dict(Counter(all_df.loc[all_df.valid, "mode"])))
print("File extensions            :", dict(Counter(Path(p).suffix.lower() for p in all_df["path"])))

bad = all_df[~all_df["valid"]]
df_train_full = df_train_full[~df_train_full["path"].isin(bad["path"])].reset_index(drop=True)
df_test       = df_test[~df_test["path"].isin(bad["path"])].reset_index(drop=True)

In [ ]:
# ───── Exact-duplicate check (MD5 of file bytes) ─────
def md5(p):
    with open(p, "rb") as f: return hashlib.md5(f.read()).hexdigest()

df_train_full["md5"] = df_train_full["path"].map(md5)
df_test["md5"]       = df_test["path"].map(md5)

n_dup_train = int(df_train_full.duplicated("md5", keep="first").sum())
n_dup_cross = int(df_test["md5"].isin(df_train_full["md5"]).sum())
print(f"Exact duplicates inside train        : {n_dup_train}")
print(f"Test images identical to a train file: {n_dup_cross}")

# Remove them: keep first copy in train; drop from test anything the model could have memorised.
df_train_full = df_train_full.drop_duplicates("md5", keep="first").reset_index(drop=True)
df_test       = df_test[~df_test["md5"].isin(df_train_full["md5"])].reset_index(drop=True)

# ───── Source-leaf overlap between train and test ─────
overlap = set(df_train_full["group"]) & set(df_test["group"])
print(f"\nSource-leaf groups shared by train and test: {len(overlap)}")
assert len(overlap) == 0, "Train/test share source photos — test accuracy would be inflated."

g = df_train_full.groupby("group").size()
print(f"Train: {len(df_train_full):,} images from {g.size:,} distinct source photos "
      f"({int((g > 1).sum())} photos appear more than once as augmented copies)")
print(f"After cleaning -> train: {len(df_train_full):,} | test: {len(df_test):,}")

**Why this matters:** train and test come from *different* source photos (checked above), so the provided `val` folder is a trustworthy held-out test set. But **inside** `train/` some source photos appear multiple times (augmented copies). A plain random validation split would put a copy in training and its sibling in validation, so we split by `group` in Section 6.

## SECTION 5 — Exploratory data analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=False)
for ax, d, title in zip(axes, [df_train_full, df_test], ["Train (after cleaning)", "Test (provided 'val' folder)"]):
    counts = d["class"].map(pretty).value_counts().reindex(pretty_names)
    sns.barplot(x=counts.values, y=counts.index, ax=ax, color="#3b8b5a")
    ax.set_title(title); ax.set_xlabel("Number of images"); ax.set_ylabel("")
    for i, v in enumerate(counts.values): ax.text(v, i, f" {v}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

imb = df_train_full["class"].value_counts()
print(f"Train class counts: min = {imb.min()}, max = {imb.max()}, imbalance ratio = {imb.max()/imb.min():.2f}")

In [ ]:
# Sample images per class (4 random per class)
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(9, 2.3 * NUM_CLASSES))
for r, c in enumerate(class_names):
    sample = df_train_full[df_train_full["class"] == c].sample(4, random_state=SEED)
    for k, p in enumerate(sample["path"]):
        axes[r, k].imshow(Image.open(p).convert("RGB")); axes[r, k].axis("off")
    axes[r, 0].set_title(pretty(c), loc="left", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Colour statistics per class (60 random images per class) — do classes differ in colour/background?
rows = []
for c in class_names:
    for p in df_train_full[df_train_full["class"] == c].sample(60, random_state=SEED)["path"]:
        a = np.asarray(Image.open(p).convert("RGB").resize(IMG_SIZE), dtype=np.float32) / 255.0
        rows.append({"class": pretty(c), "R": a[..., 0].mean(), "G": a[..., 1].mean(), "B": a[..., 2].mean(),
                     "brightness": a.mean()})
color_df = pd.DataFrame(rows)
mean_rgb = color_df.groupby("class")[["R", "G", "B"]].mean().reindex(pretty_names)
mean_rgb.plot(kind="bar", figsize=(12, 4), color=["#d9534f", "#5cb85c", "#428bca"])
plt.title("Mean channel intensity per class (sampled)"); plt.ylabel("Mean value (0-1)"); plt.xticks(rotation=40, ha="right")
plt.tight_layout(); plt.show()

**Reading the EDA**
* The class distribution plot shows whether re-weighting or resampling is required. When the ratio printed above is ≈ 1.0, classes are balanced, so plain accuracy is a fair metric and no class weights are needed. We still report **macro** precision/recall/F1 so a weak class cannot hide.
* Compare the channel-mean bars: classes whose colour profile stands out (yellowing, browning) are partly separable by colour alone, but many diseases differ mainly in lesion *texture and shape* — which is why a CNN is used rather than colour histograms.
* Sample images show a single leaf photographed on a plain grey background. Expect lower accuracy on field photos (see limitations).

## SECTION 6 — Train / validation / test split

| Set | Source | Used for |
|---|---|---|
| **Train** | ~90 % of `train/` | fitting weights |
| **Validation** | ~10 % of `train/` | early stopping, LR scheduling, model selection |
| **Test** | provided `val/` folder | one final, unbiased evaluation |

The train→validation split is **stratified by class and grouped by source photo** (`StratifiedGroupKFold`), so augmented copies of the same leaf never straddle the split.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=VAL_FRACTION_FOLDS, shuffle=True, random_state=SEED)
tr_idx, va_idx = next(sgkf.split(df_train_full, df_train_full["label"], groups=df_train_full["group"]))

df_train = df_train_full.iloc[tr_idx].reset_index(drop=True)
df_val   = df_train_full.iloc[va_idx].reset_index(drop=True)

assert len(set(df_train["group"]) & set(df_val["group"])) == 0, "Leak: same source photo in train and val"
assert len(set(df_train["group"]) & set(df_test["group"])) == 0
assert len(set(df_val["group"])   & set(df_test["group"])) == 0

split_summary = pd.DataFrame({
    "Train": df_train["class"].map(pretty).value_counts(),
    "Validation": df_val["class"].map(pretty).value_counts(),
    "Test": df_test["class"].map(pretty).value_counts(),
}).reindex(pretty_names)
split_summary.loc["TOTAL"] = split_summary.sum()
display(split_summary)
print("No source photo is shared between train, validation and test. ✔")

## SECTION 7 — Input pipeline & data augmentation

* Images are decoded lazily with `tf.data` (the full dataset never sits in RAM) and resized to 128×128.
* Pixel scaling happens **inside each model** (0-255 float in → model rescales), so the pipeline is identical for both models.
* Augmentation (flip, rotation, zoom, contrast) is applied to the **training set only**. Leaves have no canonical orientation, so flips/rotations are label-preserving.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape((*IMG_SIZE, 3))
    return tf.cast(img, tf.float32), label

augmenter = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augmenter")

def make_ds(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values.astype("int32")))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)   # shuffles cheap path strings
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.cache()                                                     # small sets: cache decoded images
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(df_train, training=True)
val_ds   = make_ds(df_val)
test_ds  = make_ds(df_test)                 # NOT shuffled -> predictions align with df_test order

xb, yb = next(iter(train_ds))
print("Batch:", xb.shape, xb.dtype, "| labels:", yb.shape, "| pixel range:", float(tf.reduce_min(xb)), "-", float(tf.reduce_max(xb)))

In [ ]:
# Visualise augmentation on one training image
p = df_train.iloc[0]["path"]
img, _ = load_image(p, 0)
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
axes[0, 0].imshow(img.numpy().astype("uint8")); axes[0, 0].set_title("Original (resized)")
for ax in axes.flat[1:]:
    aug = augmenter(tf.expand_dims(img, 0), training=True)[0].numpy()
    ax.imshow(np.clip(aug, 0, 255).astype("uint8")); ax.set_title("Augmented")
for ax in axes.flat: ax.axis("off")
plt.suptitle(pretty(df_train.iloc[0]["class"])); plt.tight_layout(); plt.show()

In [ ]:
# ───── Shared helpers ─────
def get_callbacks(name, patience=6):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, patience // 2), min_lr=1e-6, verbose=1),
        keras.callbacks.ModelCheckpoint(str(OUT_DIR / f"{name}_best.keras"), monitor="val_loss", save_best_only=True),
    ]

def merge_histories(*hists):
    out = {}
    for h in hists:
        for k, v in h.history.items(): out.setdefault(k, []).extend(v)
    return out

def plot_history(hist, title, boundary=None):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, m, lab in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
        ax.plot(hist[m], label="train"); ax.plot(hist["val_" + m], label="validation")
        if boundary is not None: ax.axvline(boundary - 0.5, color="grey", ls="--", label="fine-tuning starts")
        ax.set_title(f"{title} — {lab}"); ax.set_xlabel("Epoch"); ax.set_ylabel(lab); ax.legend()
    plt.tight_layout(); plt.show()
    best = int(np.argmin(hist["val_loss"]))
    print(f"Best epoch (lowest val_loss): {best+1} | val_acc = {hist['val_accuracy'][best]:.4f} | "
          f"train_acc = {hist['accuracy'][best]:.4f}")

## SECTION 8 — Model 1: Custom CNN (from scratch)

Architecture (an upgraded version of the supplied Conv→Pool network):

`Rescaling → [Conv3×3 → BatchNorm → ReLU → MaxPool] × 4 (32-64-128-128 filters) → GlobalAveragePooling → Dense(128) → Dropout(0.5) → Softmax(10)`

Design choices: BatchNorm stabilises training; **GlobalAveragePooling** replaces `Flatten` (far fewer parameters, less overfitting, and it keeps the spatial map that Grad-CAM needs); the last conv activation is named `last_conv_act` for explainability.

In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1.0 / 255, name="rescale")(inputs)
    for i, f in enumerate([32, 64, 128, 128], start=1):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"conv{i}")(x)
        x = layers.BatchNormalization(name=f"bn{i}")(x)
        x = layers.Activation("relu", name="last_conv_act" if i == 4 else f"relu{i}")(x)
        if i < 4:
            x = layers.MaxPooling2D(2, name=f"pool{i}")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="fc")(x)
    x = layers.Dropout(0.5, name="dropout")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")

cnn = build_custom_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

In [ ]:
t0 = time.time()
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
                   callbacks=get_callbacks("custom_cnn", patience=6), verbose=2)
time_cnn = time.time() - t0
print(f"\nTraining time: {time_cnn/60:.1f} min")
hist_cnn = merge_histories(hist_cnn)
plot_history(hist_cnn, "Custom CNN")

**How to read the curves:** a widening gap between training and validation accuracy indicates overfitting; validation loss rising while training loss falls is the signal early stopping reacts to. Because the training set is augmented and the validation set is not, validation accuracy can legitimately sit above training accuracy in early epochs.

## SECTION 9 — Model 2: Transfer learning with MobileNetV2

A network pre-trained on ImageNet already knows edges, textures and shapes. Two stages:
1. **Head training** — freeze the backbone, train only the new classifier.
2. **Fine-tuning** — unfreeze the top ~30 backbone layers with a very small learning rate (BatchNorm layers stay frozen).

> Needs internet the first time (downloads ImageNet weights). If the download fails the notebook continues with the custom CNN alone.

In [ ]:
tl_model, hist_tl, time_tl, tl_boundary = None, None, None, None
try:
    base = keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1.0 / 127.5, offset=-1, name="mnv2_preprocess")(inputs)   # MobileNetV2 expects [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
    tl_model = keras.Model(inputs, outputs, name="mobilenetv2_tl")

    # Stage 1 — head only
    tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                     loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    h1 = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL_HEAD,
                      callbacks=get_callbacks("mobilenetv2_head", patience=3), verbose=2)

    # Stage 2 — fine-tune top layers
    base.trainable = True
    for layer in base.layers[:-30]: layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization): layer.trainable = False
    tl_model.compile(optimizer=keras.optimizers.Adam(1e-5),
                     loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h2 = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL_FT,
                      callbacks=get_callbacks("mobilenetv2_ft", patience=4), verbose=2)
    time_tl = time.time() - t0
    tl_boundary = len(h1.history["loss"])
    hist_tl = merge_histories(h1, h2)
    print(f"\nTraining time: {time_tl/60:.1f} min")
    plot_history(hist_tl, "MobileNetV2", boundary=tl_boundary)
except Exception as e:
    print("⚠ Transfer-learning model skipped:", repr(e))

## SECTION 10 — Model selection & final test evaluation

The best model is chosen by **validation accuracy** (not test accuracy, to keep the test set untouched by decisions). The test set is then used once to report performance.

In [ ]:
models = {"Custom CNN": cnn}
if tl_model is not None: models["MobileNetV2 (TL)"] = tl_model
train_times = {"Custom CNN": time_cnn, "MobileNetV2 (TL)": time_tl}

y_val  = df_val["label"].values
y_test = df_test["label"].values

def evaluate_model(model, ds, y_true):
    probs = model.predict(ds, verbose=0)
    y_pred = probs.argmax(axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return probs, y_pred, {"accuracy": accuracy_score(y_true, y_pred), "macro_precision": p, "macro_recall": r, "macro_f1": f1}

results, preds = [], {}
for name, m in models.items():
    _, _, v = evaluate_model(m, val_ds, y_val)
    probs, y_pred, t = evaluate_model(m, test_ds, y_test)
    preds[name] = (probs, y_pred)
    results.append({"Model": name, "Params": m.count_params(),
                    "Train time (min)": round(train_times[name] / 60, 1),
                    "Val accuracy": v["accuracy"],
                    "Test accuracy": t["accuracy"], "Test macro-P": t["macro_precision"],
                    "Test macro-R": t["macro_recall"], "Test macro-F1": t["macro_f1"]})
results_df = pd.DataFrame(results).set_index("Model")
display(results_df.style.format({"Params": "{:,}", "Val accuracy": "{:.4f}", "Test accuracy": "{:.4f}",
                                 "Test macro-P": "{:.4f}", "Test macro-R": "{:.4f}", "Test macro-F1": "{:.4f}"}))

best_name  = results_df["Val accuracy"].idxmax()
best_model = models[best_name]
best_probs, best_pred = preds[best_name]
print(f"\nSelected model (highest validation accuracy): {best_name}")

In [ ]:
# Per-class report for the selected model
print(classification_report(y_test, best_pred, target_names=pretty_names, digits=4, zero_division=0))

In [ ]:
# Confusion matrices: counts and row-normalised (recall per class)
cm = confusion_matrix(y_test, best_pred, labels=range(NUM_CLASSES))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=pretty_names, yticklabels=pretty_names, ax=axes[0], cbar=False)
axes[0].set_title(f"{best_name} — confusion matrix (counts)")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, xticklabels=pretty_names, yticklabels=pretty_names, ax=axes[1], cbar=False)
axes[1].set_title(f"{best_name} — confusion matrix (row-normalised)")
for ax in axes: ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Per-class recall, and comparison of both models
per_class = pd.DataFrame({name: pd.Series(np.diag(confusion_matrix(y_test, p[1], labels=range(NUM_CLASSES))) /
                                            np.bincount(y_test, minlength=NUM_CLASSES), index=pretty_names)
                          for name, p in preds.items()})
per_class.plot(kind="barh", figsize=(10, 5.5))
plt.xlabel("Recall on test set"); plt.xlim(0, 1.02); plt.title("Per-class recall")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## SECTION 11 — Error analysis

In [ ]:
# Most frequent confusions (off-diagonal cells of the confusion matrix)
pairs = [(pretty_names[i], pretty_names[j], int(cm[i, j])) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
         if i != j and cm[i, j] > 0]
pairs = sorted(pairs, key=lambda t: -t[2])[:8]
print(f"Total test errors: {int((best_pred != y_test).sum())} of {len(y_test)}")
print("Top confusions (true -> predicted : count)")
for t, p_, n in pairs: print(f"  {t:35s} -> {p_:35s} : {n}")
if not pairs: print("  (no errors)")

In [ ]:
# Gallery of misclassified test images, most confident mistakes first
wrong = np.where(best_pred != y_test)[0]
wrong = wrong[np.argsort(-best_probs[wrong].max(axis=1))][:12]
if len(wrong) == 0:
    print("No misclassified test images.")
else:
    cols = 4; rows = int(np.ceil(len(wrong) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 3.6 * rows)); axes = np.atleast_1d(axes).ravel()
    for ax, i in zip(axes, wrong):
        ax.imshow(Image.open(df_test.loc[i, "path"]).convert("RGB"))
        ax.set_title(f"True: {pretty_names[y_test[i]]}\nPred: {pretty_names[best_pred[i]]} ({best_probs[i].max():.0%})",
                     fontsize=9, color="firebrick")
    for ax in axes: ax.axis("off")
    plt.suptitle("Most confident misclassifications", y=1.0); plt.tight_layout(); plt.show()

In [ ]:
# Is the model calibrated? Confidence of correct vs wrong predictions
conf = best_probs.max(axis=1); correct = best_pred == y_test
plt.figure(figsize=(8, 4))
plt.hist(conf[correct], bins=20, alpha=.7, label=f"correct (n={correct.sum()})")
if (~correct).any(): plt.hist(conf[~correct], bins=20, alpha=.7, label=f"wrong (n={(~correct).sum()})")
plt.xlabel("Predicted probability of top class"); plt.ylabel("Images"); plt.legend(); plt.title("Prediction confidence")
plt.tight_layout(); plt.show()
print(f"Mean confidence — correct: {conf[correct].mean():.3f}" + (f" | wrong: {conf[~correct].mean():.3f}" if (~correct).any() else ""))

## SECTION 12 — Explainability with Grad-CAM

Grad-CAM highlights the image regions that pushed the network towards its predicted class. For a disease detector this is a sanity check: heat should sit on lesions, spots, mould or discoloured tissue — not on the background or leaf edge. It is computed here for the **custom CNN** (its last conv activation is named `last_conv_act`).

In [ ]:
def make_gradcam(img_batch, model, conv_layer="last_conv_act", pred_index=None):
    grad_model = keras.Model(model.inputs, [model.get_layer(conv_layer).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch, training=False)
        if pred_index is None: pred_index = int(tf.argmax(preds[0]))
        score = preds[:, pred_index]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    heat = tf.squeeze(conv_out[0] @ weights[..., tf.newaxis])
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy(), pred_index, preds[0].numpy()

def overlay(img_uint8, heat, alpha=0.45):
    h = np.asarray(Image.fromarray((heat * 255).astype("uint8")).resize(IMG_SIZE, Image.BILINEAR)) / 255.0
    colored = plt.get_cmap("jet")(h)[..., :3]
    return np.clip((1 - alpha) * (img_uint8 / 255.0) + alpha * colored, 0, 1)

sample_idx = df_test.groupby("label").sample(1, random_state=SEED).index.tolist()   # one test image per class
fig, axes = plt.subplots(2, len(sample_idx), figsize=(2.4 * len(sample_idx), 5.4))
for k, i in enumerate(sample_idx):
    img, _ = load_image(df_test.loc[i, "path"], 0)
    heat, pi, pr = make_gradcam(tf.expand_dims(img, 0), cnn)
    axes[0, k].imshow(img.numpy().astype("uint8")); axes[0, k].axis("off")
    axes[0, k].set_title(f"True: {pretty_names[y_test[i]][:16]}", fontsize=8)
    axes[1, k].imshow(overlay(img.numpy(), heat)); axes[1, k].axis("off")
    axes[1, k].set_title(f"Pred: {pretty_names[pi][:16]}\n{pr[pi]:.0%}", fontsize=8,
                         color="green" if pi == y_test[i] else "red")
plt.suptitle("Grad-CAM — custom CNN (top: input, bottom: attention)", y=1.02); plt.tight_layout(); plt.show()

## SECTION 13 — Save artefacts & inference helper

In [ ]:
best_model.save(OUT_DIR / "tomato_disease_best_model.keras")
with open(OUT_DIR / "class_names.json", "w") as f: json.dump({"class_names": class_names, "img_size": IMG_SIZE}, f, indent=2)
results_df.to_csv(OUT_DIR / "model_comparison.csv")
print("Saved to:", OUT_DIR.resolve()); print(sorted(p.name for p in OUT_DIR.iterdir()))

In [ ]:
def predict_image(path, model=None, top_k=3, show=True):
    """Predict the disease class for one leaf photo (any size, RGB or grayscale)."""
    model = model or best_model
    img, _ = load_image(str(path), 0)
    probs = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    top = probs.argsort()[::-1][:top_k]
    if show:
        plt.figure(figsize=(3.5, 3.8)); plt.imshow(img.numpy().astype("uint8")); plt.axis("off")
        plt.title(f"{pretty_names[top[0]]}\n({probs[top[0]]:.1%})"); plt.show()
    return [(pretty_names[i], float(probs[i])) for i in top]

demo = df_test.sample(1, random_state=7).iloc[0]
print("True label:", pretty(demo["class"]))
for name, p in predict_image(demo["path"]): print(f"  {name:32s} {p:.1%}")

# To reload later in a new session:
# model = keras.models.load_model("outputs/tomato_disease_best_model.keras")

### 13b — Export for the Streamlit dashboard
The next two cells write everything the web app needs into `project/` (create it next to this notebook): the trained model, class names, real evaluation results, and a dataset manifest. **The app never retrains and never invents numbers — it only reads these files.**

In [ ]:
# ───── (a) Dataset artefacts: manifest + audit summary ─────
PROJECT_DIR = Path('project')
ROOT_DATA = Path('data'); ROOT_DATA.mkdir(exist_ok=True)
(PROJECT_DIR / 'model').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'data').mkdir(parents=True, exist_ok=True)

manifest = pd.concat([df_train.assign(split='train'), df_val.assign(split='validation'),
                      df_test.assign(split='test')], ignore_index=True)
manifest['path'] = manifest['path'].str.replace('\\', '/', regex=False)
manifest['class_pretty'] = manifest['class'].map(pretty)
manifest = manifest[['path', 'class', 'class_pretty', 'label', 'split', 'group']]
manifest.to_csv(PROJECT_DIR / 'data' / 'dataset.csv', index=False)
manifest.to_csv(ROOT_DATA / 'dataset.csv', index=False)

_grp = df_train_full.groupby('group').size()
audit = {
    'files_scanned': int(len(all_df)),
    'corrupt_files': int((~all_df['valid']).sum()),
    'image_sizes': {f'{w}x{h}': int(n) for (w, h), n in Counter(all_df.loc[all_df.valid, 'size']).items()},
    'colour_modes': dict(Counter(all_df.loc[all_df.valid, 'mode'])),
    'exact_duplicates_removed_from_train': int(n_dup_train),
    'test_images_removed_as_train_duplicates': int(n_dup_cross),
    'train_source_photos': int(_grp.size),
    'source_photos_with_augmented_copies': int((_grp > 1).sum()),
    'source_photos_shared_train_vs_test': int(len(overlap)),
}
with open(PROJECT_DIR / 'data' / 'audit.json', 'w') as f: json.dump(audit, f, indent=2)
with open(ROOT_DATA / 'audit.json', 'w') as f: json.dump(audit, f, indent=2)
print('Wrote', PROJECT_DIR / 'data' / 'dataset.csv', f'({len(manifest):,} rows)')


In [ ]:
# ───── (b) Model artefacts: model + class names + REAL evaluation results ─────
from sklearn.metrics import roc_auc_score
import shutil

ROOT_MODEL = Path('model'); ROOT_MODEL.mkdir(exist_ok=True)
best_model.save(PROJECT_DIR / 'model' / 'model.keras')
shutil.copyfile(PROJECT_DIR / 'model' / 'model.keras', ROOT_MODEL / 'model.keras')

class_meta = {
    'class_names': class_names, 'pretty_names': pretty_names, 'img_size': list(IMG_SIZE),
    'gradcam_layer': 'last_conv_act' if best_name == 'Custom CNN' else None
}
with open(PROJECT_DIR / 'model' / 'class_names.json', 'w') as f: json.dump(class_meta, f, indent=2)
with open(ROOT_MODEL / 'class_names.json', 'w') as f: json.dump(class_meta, f, indent=2)

try:    auc = float(roc_auc_score(y_test, best_probs, multi_class='ovr', average='macro'))
except Exception: auc = None
_, _, best_scores = evaluate_model(best_model, test_ds, y_test)

histories = {'Custom CNN': hist_cnn}
if hist_tl is not None: histories['MobileNetV2 (TL)'] = hist_tl

metrics = {
    'model_name': best_name,
    'framework': f'TensorFlow {tf.__version__}',
    'num_parameters': int(best_model.count_params()),
    'test': {**{k: float(v) for k, v in best_scores.items()}, 'roc_auc_macro_ovr': auc, 'n_images': int(len(y_test))},
    'class_names': pretty_names,
    'per_class': classification_report(y_test, best_pred, target_names=pretty_names, output_dict=True, zero_division=0),
    'confusion_matrix': confusion_matrix(y_test, best_pred, labels=range(NUM_CLASSES)).tolist(),
    'comparison': results_df.reset_index().to_dict(orient='records'),
    'histories': {k: {m: [float(x) for x in v] for m, v in h.items() if m in ('accuracy', 'val_accuracy', 'loss', 'val_loss')}
                  for k, h in histories.items()},
    'split_sizes': {'train': int(len(df_train)), 'validation': int(len(df_val)), 'test': int(len(df_test))},
    'config': {'seed': SEED, 'img_size': list(IMG_SIZE), 'batch_size': BATCH_SIZE},
}
with open(PROJECT_DIR / 'model' / 'metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
with open(ROOT_MODEL / 'metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
print('Exported to project/ and model/:', sorted(str(p.name) for p in (PROJECT_DIR / 'model').iterdir()))


## SECTION 14 — Conclusion, limitations & future work

In [ ]:
best_row = results_df.loc[best_name]
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Dataset          : {len(df_train):,} train / {len(df_val):,} validation / {len(df_test):,} test images, {NUM_CLASSES} classes")
print(f"Selected model   : {best_name}  ({int(best_row['Params']):,} parameters)")
print(f"Test accuracy    : {best_row['Test accuracy']:.4f}")
print(f"Test macro-F1    : {best_row['Test macro-F1']:.4f}")
worst = pd.Series(np.diag(cm) / cm.sum(axis=1), index=pretty_names).sort_values()
print(f"Weakest classes  : " + ", ".join(f"{n} ({v:.0%})" for n, v in worst.head(3).items()))
print("=" * 70)

### Conclusions
* A CNN trained on the PlantVillage tomato images classifies the 10 leaf conditions; the exact scores are in the summary above and in `outputs/model_comparison.csv`.
* The evaluation is deliberately strict: exact duplicates were removed, the test set shares **no source photo** with training, and the validation split was **grouped by source photo**, so scores are not inflated by augmented look-alikes.
* Comparing the from-scratch CNN with the pre-trained MobileNetV2 shows the value (or not) of transfer learning on this dataset — see the results table.
* Grad-CAM and the error analysis show *what* the model looks at and *which* diseases it confuses.

### Limitations
* **Domain shift.** PlantVillage photos are taken in controlled conditions (single leaf, plain background). Accuracy on field photos with clutter, shadows and mixed symptoms is expected to be noticeably lower.
* Only tomato leaves and only single-disease images; multiple simultaneous diseases are not modelled.
* One train/validation split and one random seed — no confidence intervals.
* Image size 128×128 is a speed/accuracy compromise; small early lesions may be lost.

### Future work
* Test on in-field images (e.g. PlantDoc) and add field-style augmentation (background clutter, lighting, blur).
* Multiple seeds / k-fold (grouped) to report mean ± std.
* Larger backbones (EfficientNet, ConvNeXt), higher resolution, and test-time augmentation.
* Probability calibration and an "uncertain / not a tomato leaf" rejection option before any real-world use.
* Export to TFLite for on-device use by farmers.